# 01 – Eksploracyjna Analiza Danych (EDA)
**Dataset:** Heart Disease UCI – Cleveland

Celem tego notebooka jest zrozumienie struktury danych przed budowaniem modelu.

## 1. Import bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 2. Wczytanie danych

In [ ]:
df = pd.read_csv('../data/heart.csv')
print(f"Rozmiar: {df.shape[0]} wierszy x {df.shape[1]} kolumn")
df.head(10)

## 3. Podstawowe informacje

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 4. Brakujące wartości
Dataset Cleveland używa `?` jako brakujące wartości.

In [ ]:
df_clean = df.replace('?', np.nan)
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(2)
pd.DataFrame({'Brakujące': missing, 'Procent (%)': missing_pct})[missing > 0]

## 5. Rozkład zmiennej docelowej (target)
`0` = brak choroby, `1–4` = choroba serca (sprowadzamy do binarnego)

In [ ]:
df_clean = df_clean.astype(float)
df_clean['target_bin'] = (df_clean['target'] > 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Oryginalne wartości
df_clean['target'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', alpha=0.8, edgecolor='white'
)
axes[0].set_title('Oryginalne wartości target (0–4)')
axes[0].set_xlabel('Wartość target')
axes[0].set_ylabel('Liczba próbek')

# Binarne
labels = ['Brak choroby (0)', 'Choroba (1)']
counts = df_clean['target_bin'].value_counts().sort_index()
axes[1].pie(counts, labels=labels, autopct='%1.1f%%',
            colors=['#5B9BD5', '#ED7D31'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Binarne: choroba vs brak choroby')

plt.tight_layout()
plt.show()

print(f"Brak choroby: {counts[0]} ({counts[0]/len(df_clean)*100:.1f}%)")
print(f"Choroba:      {counts[1]} ({counts[1]/len(df_clean)*100:.1f}%)")

## 6. Rozkłady cech numerycznych

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, col in enumerate(num_cols):
    # Histogram
    df_clean[col].hist(ax=axes[0, i], bins=20, color='steelblue',
                       alpha=0.8, edgecolor='white')
    axes[0, i].set_title(col)
    axes[0, i].set_ylabel('Liczba')

    # Box plot wg target
    data_0 = df_clean.loc[df_clean['target_bin'] == 0, col].dropna()
    data_1 = df_clean.loc[df_clean['target_bin'] == 1, col].dropna()
    axes[1, i].boxplot([data_0, data_1], labels=['Brak', 'Choroba'],
                       patch_artist=True,
                       boxprops={'facecolor': '#5B9BD5', 'alpha': 0.6})
    axes[1, i].set_title(f'{col} vs target')

plt.suptitle('Rozkłady cech numerycznych', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Cechy kategoryczne vs target

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
cat_labels = {
    'sex':     {0: 'Kobieta', 1: 'Mężczyzna'},
    'cp':      {0: 'Typowy', 1: 'Atypowy', 2: 'Nieangina', 3: 'Bezob.'},
    'fbs':     {0: '≤120', 1: '>120'},
    'restecg': {0: 'Normalny', 1: 'ST-T', 2: 'LVH'},
    'exang':   {0: 'Nie', 1: 'Tak'},
    'slope':   {0: 'Rosnący', 1: 'Płaski', 2: 'Malejący'},
}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = df_clean.groupby([col, 'target_bin']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=['#5B9BD5', '#ED7D31'],
            alpha=0.85, edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].legend(['Brak choroby', 'Choroba'], fontsize=8)
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Cechy kategoryczne a choroba serca', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Macierz korelacji

In [ ]:
df_corr = df_clean.drop('target', axis=1).copy()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(df_corr.corr(), dtype=bool))
sns.heatmap(df_corr.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Macierz korelacji Pearsona', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

## 9. Wnioski z EDA

- **Rozkład klas**: ~54% brak choroby, ~46% choroba – klasy są względnie zrównoważone
- **Najsilniejsza korelacja z target**: `cp` (ból w klatce), `thalach` (max tętno), `exang` (dławica wysiłkowa), `oldpeak`
- **Brakujące wartości**: tylko w `ca` i `thal` (4–6 próbek) – uzupełnione medianą
- **Outliers**: `chol` ma kilka ekstremów, ale RF jest odporny na wartości skrajne